In [ ]:
from google.colab import drive
import os
import json
import joblib
import numpy as np
import pandas as pd

drive.mount("/content/drive")

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

feature_df = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_phase5_feature_engineered.parquet"
    )
)

feature_df["Datetime"] = pd.to_datetime(
    feature_df["Datetime"]
)

feature_df = feature_df.sort_values(
    ["StationId", "Datetime"]
).reset_index(drop=True)

print("Dataset shape:", feature_df.shape)
print("Stations:", feature_df["StationId"].nunique())

In [ ]:
unique_times = np.sort(
    feature_df["Datetime"].unique()
)

split_position = int(
    len(unique_times) * 0.80
)

train_end_time = unique_times[
    split_position - 1
]

train_df = feature_df[
    feature_df["Datetime"] <= train_end_time
].copy()

test_df = feature_df[
    feature_df["Datetime"] > train_end_time
].copy()

print("Training end:", train_end_time)
print(
    "Test starts:",
    test_df["Datetime"].min()
)

print(
    "Train rows:",
    len(train_df)
)

print(
    "Test rows:",
    len(test_df)
)

In [ ]:
dl_numeric_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene",
    "Hour",
    "Day",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

categorical_feature = "StationId"
target = "AQI_t_plus_1"

print(
    "Numeric features:",
    len(dl_numeric_features)
)

print(
    "Categorical feature:",
    categorical_feature
)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

numeric_imputer = SimpleImputer(
    strategy="median"
)

numeric_scaler = StandardScaler()

train_numeric = numeric_imputer.fit_transform(
    train_df[dl_numeric_features]
)

train_numeric = numeric_scaler.fit_transform(
    train_numeric
)

all_numeric = numeric_imputer.transform(
    feature_df[dl_numeric_features]
)

all_numeric = numeric_scaler.transform(
    all_numeric
)

station_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

train_station = station_encoder.fit_transform(
    train_df[[categorical_feature]]
)

all_station = station_encoder.transform(
    feature_df[[categorical_feature]]
)

print(
    "Numeric shape:",
    all_numeric.shape
)

print(
    "Station encoding shape:",
    all_station.shape
)

In [ ]:
all_processed = np.hstack(
    [
        all_numeric,
        all_station
    ]
).astype(
    "float32"
)

print(
    "Complete processed feature matrix:",
    all_processed.shape
)

In [ ]:
LOOKBACK = 12

def build_live_style_test_sequences(
    df,
    processed_matrix,
    test_start,
    lookback,
    target_column
):
    sequences = []
    targets = []
    metadata = []

    station_values = df["StationId"].astype(str).to_numpy()
    timestamps = pd.to_datetime(
        df["Datetime"]
    ).to_numpy()

    target_values = df[
        target_column
    ].to_numpy(
        dtype="float32"
    )

    test_start = pd.Timestamp(
        test_start
    )

    station_ids = np.unique(
        station_values
    )

    for station_id in station_ids:

        positions = np.where(
            station_values == station_id
        )[0]

        station_positions = positions[
            np.argsort(
                timestamps[positions]
            )
        ]

        station_times = timestamps[
            station_positions
        ]

        for local_i in range(
            lookback,
            len(station_positions)
        ):

            current_position = (
                station_positions[local_i]
            )

            current_time = pd.Timestamp(
                station_times[local_i]
            )

            # Only evaluate the future test period.
            if current_time <= test_start:
                continue

            window_positions = station_positions[
                local_i - lookback:local_i
            ]

            window_times = timestamps[
                window_positions
            ]

            # Require 12 consecutive hourly observations.
            time_diffs = np.diff(
                window_times.astype(
                    "datetime64[s]"
                ).astype(
                    np.int64
                )
            )

            hourly_seconds = 3600

            if not np.all(
                time_diffs == hourly_seconds
            ):
                continue

            # Current t -> target at t+1.
            target_value = target_values[
                current_position
            ]

            if not np.isfinite(
                target_value
            ):
                continue

            sequences.append(
                processed_matrix[
                    window_positions
                ]
            )

            targets.append(
                target_value
            )

            metadata.append(
                {
                    "StationId": station_id,
                    "Datetime": current_time
                }
            )

    return (
        np.asarray(
            sequences,
            dtype="float32"
        ),
        np.asarray(
            targets,
            dtype="float32"
        ),
        pd.DataFrame(metadata)
    )

X_test_dl, y_test_dl, test_metadata_dl = (
    build_live_style_test_sequences(
        feature_df,
        all_processed,
        test_df["Datetime"].min(),
        LOOKBACK,
        target
    )
)

print(
    "Standardized test sequences:",
    X_test_dl.shape
)

print(
    "Targets:",
    y_test_dl.shape
)

print(
    "Stations represented:",
    test_metadata_dl["StationId"].nunique()
)

In [ ]:
stations_in_test = sorted(
    test_metadata_dl["StationId"]
    .unique()
)

print(
    "Stations in standardized DL test:",
    len(stations_in_test)
)

print(stations_in_test)

In [ ]:
first_row = (
    test_metadata_dl
    .sort_values("Datetime")
    .iloc[0]
)

print(
    "First standardized test prediction time:",
    first_row["Datetime"]
)

print(
    "Its station:",
    first_row["StationId"]
)

print(
    "\nThis sequence uses the preceding",
    LOOKBACK,
    "hours available at prediction time."
)

In [ ]:
import tensorflow as tf

lstm_model = tf.keras.models.load_model(
    os.path.join(
        MODEL_PATH,
        "phase8_lstm.keras"
    )
)

cnn_bilstm_model = tf.keras.models.load_model(
    os.path.join(
        MODEL_PATH,
        "phase8_cnn_bilstm.keras"
    )
)

print("LSTM loaded.")
print("CNN-BiLSTM loaded.")

In [ ]:
print(
    "Standardized sequence shape:",
    X_test_dl.shape
)

print(
    "LSTM expected input:",
    lstm_model.input_shape
)

print(
    "CNN-BiLSTM expected input:",
    cnn_bilstm_model.input_shape
)

In [ ]:
lstm_standard_pred = (
    lstm_model
    .predict(
        X_test_dl,
        batch_size=256,
        verbose=1
    )
    .ravel()
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

lstm_standard_mae = mean_absolute_error(
    y_test_dl,
    lstm_standard_pred
)

lstm_standard_rmse = np.sqrt(
    mean_squared_error(
        y_test_dl,
        lstm_standard_pred
    )
)

lstm_standard_r2 = r2_score(
    y_test_dl,
    lstm_standard_pred
)

print("STANDARDIZED LSTM")
print("-----------------")
print("MAE :", round(lstm_standard_mae, 6))
print("RMSE:", round(lstm_standard_rmse, 6))
print("R²  :", round(lstm_standard_r2, 6))

In [ ]:
cnn_standard_pred = (
    cnn_bilstm_model
    .predict(
        X_test_dl,
        batch_size=256,
        verbose=1
    )
    .ravel()
)

cnn_standard_mae = mean_absolute_error(
    y_test_dl,
    cnn_standard_pred
)

cnn_standard_rmse = np.sqrt(
    mean_squared_error(
        y_test_dl,
        cnn_standard_pred
    )
)

cnn_standard_r2 = r2_score(
    y_test_dl,
    cnn_standard_pred
)

print("STANDARDIZED CNN-BiLSTM")
print("-----------------------")
print("MAE :", round(cnn_standard_mae, 6))
print("RMSE:", round(cnn_standard_rmse, 6))
print("R²  :", round(cnn_standard_r2, 6))

In [ ]:
classical_results = pd.read_csv(
    os.path.join(
        TABLE_PATH,
        "phase6_classical_model_results.csv"
    )
)

rf_row = classical_results[
    classical_results["Model"]
    == "Random Forest"
].iloc[0]

rf_mae = rf_row["MAE"]
rf_rmse = rf_row["RMSE"]
rf_r2 = rf_row["R2"]

print("Random Forest")
print("MAE :", round(rf_mae, 6))
print("RMSE:", round(rf_rmse, 6))
print("R²  :", round(rf_r2, 6))

In [ ]:
standardized_results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "LSTM",
        "CNN-BiLSTM"
    ],
    "MAE": [
        rf_mae,
        lstm_standard_mae,
        cnn_standard_mae
    ],
    "RMSE": [
        rf_rmse,
        lstm_standard_rmse,
        cnn_standard_rmse
    ],
    "R2": [
        rf_r2,
        lstm_standard_r2,
        cnn_standard_r2
    ],
    "EvaluationScope": [
        "Full Delhi future test period",
        "Full Delhi future test period - live-style sequences",
        "Full Delhi future test period - live-style sequences"
    ],
    "Stations": [
        37,
        test_metadata_dl["StationId"].nunique(),
        test_metadata_dl["StationId"].nunique()
    ]
})

standardized_results = (
    standardized_results
    .sort_values("RMSE")
    .reset_index(drop=True)
)

display(standardized_results)

In [ ]:
standardized_results[
    "RMSE_vs_RF_%"
] = (
    (
        standardized_results["RMSE"]
        - rf_rmse
    )
    / rf_rmse
) * 100

standardized_results[
    "MAE_vs_RF_%"
] = (
    (
        standardized_results["MAE"]
        - rf_mae
    )
    / rf_mae
) * 100

display(
    standardized_results
)

In [ ]:
lstm_eval = test_metadata_dl.copy()

lstm_eval["Actual"] = y_test_dl
lstm_eval["Prediction"] = lstm_standard_pred

lstm_station_results = (
    lstm_eval
    .groupby("StationId")
    .apply(
        lambda g: pd.Series({
            "Records": len(g),
            "MAE": mean_absolute_error(
                g["Actual"],
                g["Prediction"]
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    g["Actual"],
                    g["Prediction"]
                )
            ),
            "R2": r2_score(
                g["Actual"],
                g["Prediction"]
            )
        })
    )
    .sort_values("RMSE")
)

display(lstm_station_results)

In [ ]:
cnn_eval = test_metadata_dl.copy()

cnn_eval["Actual"] = y_test_dl
cnn_eval["Prediction"] = cnn_standard_pred

cnn_station_results = (
    cnn_eval
    .groupby("StationId")
    .apply(
        lambda g: pd.Series({
            "Records": len(g),
            "MAE": mean_absolute_error(
                g["Actual"],
                g["Prediction"]
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    g["Actual"],
                    g["Prediction"]
                )
            ),
            "R2": r2_score(
                g["Actual"],
                g["Prediction"]
            )
        })
    )
    .sort_values("RMSE")
)

display(cnn_station_results)

In [ ]:
test_actual_lookup = feature_df[
    [
        "StationId",
        "Datetime",
        "AQI"
    ]
].drop_duplicates(
    ["StationId", "Datetime"]
)

lstm_eval = lstm_eval.merge(
    test_actual_lookup,
    on=["StationId", "Datetime"],
    how="left",
    validate="one_to_one"
)

cnn_eval = cnn_eval.merge(
    test_actual_lookup,
    on=["StationId", "Datetime"],
    how="left",
    validate="one_to_one"
)

extreme_threshold = np.quantile(
    test_actual_lookup["AQI"].dropna(),
    0.90
)

lstm_extreme = lstm_eval[
    lstm_eval["AQI"] >= extreme_threshold
]

cnn_extreme = cnn_eval[
    cnn_eval["AQI"] >= extreme_threshold
]

print(
    "Extreme AQI threshold:",
    extreme_threshold
)

print("\nLSTM extreme-period RMSE:")
print(
    np.sqrt(
        mean_squared_error(
            lstm_extreme["Actual"],
            lstm_extreme["Prediction"]
        )
    )
)

print("\nCNN-BiLSTM extreme-period RMSE:")
print(
    np.sqrt(
        mean_squared_error(
            cnn_extreme["Actual"],
            cnn_extreme["Prediction"]
        )
    )
)

In [ ]:
standardized_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase11_5_standardized_model_results.csv"
    ),
    index=False
)

lstm_station_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase11_5_lstm_station_results.csv"
    )
)

cnn_station_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase11_5_cnn_bilstm_station_results.csv"
    )
)

print("Standardized validation results saved.")

In [ ]:
best_standardized = (
    standardized_results
    .sort_values("RMSE")
    .iloc[0]
)

print("=" * 60)
print("PHASE 11.5 — STANDARDIZED MODEL RESULT")
print("=" * 60)

print(
    "Best model:",
    best_standardized["Model"]
)

print(
    "MAE:",
    round(
        best_standardized["MAE"],
        6
    )
)

print(
    "RMSE:",
    round(
        best_standardized["RMSE"],
        6
    )
)

print(
    "R²:",
    round(
        best_standardized["R2"],
        6
    )
)

print(
    "\nEvaluation:",
    best_standardized["EvaluationScope"]
)

print("=" * 60)

In [ ]:
phase11_5_decision = {
    "completed_phase": "Phase 11.5 - Standardized Full-Delhi Deep-Learning Validation",

    "best_standardized_model": str(
        best_standardized["Model"]
    ),

    "best_mae": float(
        best_standardized["MAE"]
    ),

    "best_rmse": float(
        best_standardized["RMSE"]
    ),

    "best_r2": float(
        best_standardized["R2"]
    ),

    "random_forest_rmse": float(
        rf_rmse
    ),

    "random_forest_remains_best": (
        best_standardized["Model"]
        == "Random Forest"
    ),

    "next_phase": (
        "Phase 12 - Reusable Live-Style Inference Pipeline"
    )
}

decision_path = os.path.join(
    OUTPUT_PATH,
    "phase11_5_decision.json"
)

with open(
    decision_path,
    "w"
) as f:
    json.dump(
        phase11_5_decision,
        f,
        indent=4
    )

print(
    "Phase 11.5 decision saved:"
)

print(decision_path)

In [ ]:
session_status = {
    "completed_phase": "Phase 11.5 - Standardized Full-Delhi Deep-Learning Validation",
    "next_phase": "Phase 12 - Reusable Live-Style Inference Pipeline",
    "best_standardized_model": str(
        best_standardized["Model"]
    )
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(
    status_path,
    "w"
) as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

print("Phase 11.5 status saved.")

In [ ]:
import os
import json

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

# --------------------------------------------------
# 1. Save standardized comparison
# --------------------------------------------------

standardized_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase11_5_standardized_model_results.csv"
    ),
    index=False
)

# --------------------------------------------------
# 2. Save station-level evaluation
# --------------------------------------------------

lstm_station_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase11_5_lstm_station_results.csv"
    )
)

cnn_station_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase11_5_cnn_bilstm_station_results.csv"
    )
)

# --------------------------------------------------
# 3. Save final standardized decision
# --------------------------------------------------

phase11_5_decision = {
    "completed_phase": "Phase 11.5 - Standardized Full-Delhi Deep-Learning Validation",
    "next_phase": "Phase 12 - Reusable Live-Style Inference Pipeline",

    "best_model": "Random Forest",

    "mae": 0.014194,
    "rmse": 0.731606,
    "r2": 0.999966,

    "evaluation_scope": "Full Delhi future test period - 37 stations",

    "status": "Production candidate",

    "final_model_selected": False,

    "reason": (
        "Random Forest remains the best model under the "
        "standardized full-Delhi future-test protocol. "
        "Final production selection requires live-style "
        "inference validation."
    )
}

decision_path = os.path.join(
    OUTPUT_PATH,
    "phase11_5_decision.json"
)

with open(
    decision_path,
    "w"
) as f:
    json.dump(
        phase11_5_decision,
        f,
        indent=4
    )

# --------------------------------------------------
# 4. Save session status
# --------------------------------------------------

session_status = {
    "completed_phase": "Phase 11.5 - Standardized Full-Delhi Deep-Learning Validation",
    "next_phase": "Phase 12 - Reusable Live-Style Inference Pipeline",
    "current_best_model": "Random Forest",
    "final_model_selected": False
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(
    status_path,
    "w"
) as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

# --------------------------------------------------
# 5. Verify
# --------------------------------------------------

files_to_check = [
    os.path.join(
        TABLE_PATH,
        "phase11_5_standardized_model_results.csv"
    ),
    os.path.join(
        TABLE_PATH,
        "phase11_5_lstm_station_results.csv"
    ),
    os.path.join(
        TABLE_PATH,
        "phase11_5_cnn_bilstm_station_results.csv"
    ),
    decision_path,
    status_path
]

print("=" * 60)
print("PHASE 11.5 CHECKPOINT SAVED")
print("=" * 60)

for file in files_to_check:
    print(
        "OK   " if os.path.exists(file) else "ERROR",
        file
    )

print("\nCurrent production candidate: Random Forest")
print("Next: Phase 12 - Reusable Live-Style Inference Pipeline")